# Estructura del dataset

- Session 1: Neutral, Smile
- Session 2: Neutral, surprise, squint
- Session 3: Neutral, smile, disgust
- Session 4: Neutral, Scream, Neutral


In [2]:
import pandas as pd

df = pd.read_csv("/home12TB1/database/recognition/faces/MultiPie/demographic_info_cropped.csv")
print("Unique recordingids:", df['recording_id'].unique())
print("Unique Session IDs", df['session_id'].unique())

Unique recordingids: [2 1 3]
Unique Session IDs ['session01' 'session02' 'session03' 'session04']


# Estadísticas del dataset

Número de usuarios: `748120`

In [10]:
print("Unique user id", df["subject_id"])
print("Number of cameras: ", len(df["camera_id"].unique()))
print("Pics of user 22:", len(df[df["subject_id"] == 22 ])/15)

Unique user id 0         155
1         155
2         155
3         155
4         155
         ... 
748115    342
748116    342
748117    342
748118    342
748119    342
Name: subject_id, Length: 748120, dtype: int64
Number of cameras:  15
Pics of user 22: 218.93333333333334


In [ ]:
print(df[df["subject_id"] == 22].groupby(['session_id', 'recording_id']).size())

session_id  recording_id
session01   1               300
            2               300
session02   1               300
            2               300
            3               300
session03   1               297
            2               300
            3               287
session04   1               300
            2               300
            3               300
dtype: int64


# Número de hombres y mujeres

In [14]:
print("Numero de hombres", len(df[df["gender"] == "Male"]))
print("Numero de mujeres", len(df[df["gender"] == "Female"]))


Numero de hombres 535793
Numero de mujeres 212327


# Numero de hombres y mujeres para las 6 expresiones

In [6]:
import pandas as pd

# 1. Cargar el CSV
df = pd.read_csv("/home12TB1/database/recognition/faces/MultiPie/demographic_info_cropped.csv")

# 2. Función de mapeo corregida
def map_expression(row):
    # Aseguramos que tratamos con strings y enteros limpios
    s = str(row['session_id']).lower() # Convertimos a minúsculas por si acaso
    r = int(row['recording_id'])

    if r == 1: return 0 # Neutral
    
    # Usamos "01" o "session01" dependiendo de cómo venga en tu CSV
    if '01' in s and r == 2: return 1 # Smile
    if '02' in s and r == 2: return 2 # Surprise
    if '02' in s and r == 3: return 3 # Squint
    if '03' in s and r == 2: return 1 # Smile
    if '03' in s and r == 3: return 4 # Disgust
    if '04' in s and r == 2: return 5 # Scream
    if '04' in s and r == 3: return 0 # Neutral
    return -1

# 3. Aplicar
df['expression'] = df.apply(map_expression, axis=1)

# 4. Filtrar y limpiar
df_valid = df[df['expression'] != -1].copy()

# 5. RESULTADOS
print("--- Tabla de Contingencia ---")
# Añadimos margins=True para ver los totales de fila y columna de un vistazo
contingency_table = pd.crosstab(df_valid['expression'], df_valid['gender'], margins=True)
print(contingency_table)

print("\n--- Distribución de Expresiones ---")
# Mapeo de nombres para que sea más fácil de leer
exp_names = {0: "Neutral", 1: "Smile", 2: "Surprise", 3: "Squint", 4: "Disgust", 5: "Scream"}
counts = df_valid['expression'].value_counts().sort_index()
for idx, count in counts.items():
    print(f"{exp_names[idx]}: {count}")

--- Tabla de Contingencia ---
gender      Female    Male     All
expression                        
0            98390  246601  344991
1            40667  101992  142659
2            17105   43617   60722
3            16534   43814   60348
4            19177   49123   68300
5            20454   50646   71100
All         212327  535793  748120

--- Distribución de Expresiones ---
Neutral: 344991
Smile: 142659
Surprise: 60722
Squint: 60348
Disgust: 68300
Scream: 71100


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths (Update these to your actual paths)
CSV_PATH = "/home12TB1/database/recognition/faces/MultiPie/demographic_info_cropped.csv"

def count_squint_girls(csv_path):
    df = pd.read_csv(csv_path)
    
    # 1. Replicate your label generation
    def map_row(row):
        session = str(row['session_id'])
        rec = row['recording_id']
        if 'session02' in session and rec == 3:
            return 3  # Squint
        return -1

    df['temp_label'] = df.apply(map_row, axis=1)
    df = df[df['temp_label'] == 3] # Filter only Squint

    # 2. Replicate your Subject-Disjoint Split (70% Train)
    subjects_df = df[['subject_id', 'gender']].drop_duplicates()
    
    train_subs, _ = train_test_split(
        subjects_df,
        test_size=0.3,
        stratify=subjects_df['gender'],
        random_state=42
    )

    # 3. Filter the main dataframe for training subjects
    train_df = df[df['subject_id'].isin(train_subs['subject_id'])]

    # 4. Count Females
    girls_squint = train_df[train_df['gender'] == 'Female']
    
    print(f"--- Squint Analysis ---")
    print(f"Total Girls with Squint in Train Set: {len(girls_squint)}")
    print(f"Total Unique Female Subjects in Train Set: {train_subs[train_subs['gender']=='Female'].shape[0]}")
    
    return len(girls_squint)

if __name__ == "__main__":
    count_squint_girls(CSV_PATH)

--- Squint Analysis ---
Total Girls with Squint in Train Set: 14759
Total Unique Female Subjects in Train Set: 111
